Scraper

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote
import os, time, re, json

BASE_URL = "https://angansweets.com"

# All 14 category URLs found on the homepage
CATEGORY_URLS = [
    "https://angansweets.com/products?categoryId=56aa3726-61f9-494d-a8db-4bb459ee6a0e",  # SWEETS
    "https://angansweets.com/products?categoryId=939e21e4-f220-4eaf-9ac8-906b3b71f290",  # SWEET BOX
    "https://angansweets.com/products?categoryId=d4943e69-47a6-4f53-84fa-42b05d805cca",  # Dry Fruits
    "https://angansweets.com/products?categoryId=39ee2b04-7c15-4589-b9de-5299b099327f",  # COMBOS
    "https://angansweets.com/products?categoryId=d7938f15-9233-4c6b-b59c-1cd163146495",  # FAST FOODS
    "https://angansweets.com/products?categoryId=bed5fbf7-92de-4cef-ba1e-ff79b715fd56",  # SOUTH INDIAN
    "https://angansweets.com/products?categoryId=7fa290f5-2420-4d34-bd7a-2fef6b62cd60",  # NORTH INDIAN
    "https://angansweets.com/products?categoryId=b111bcde-c6f5-49bd-abee-1d7e27d15a9b",  # TAWA & TANDOOR
    "https://angansweets.com/products?categoryId=7e9abee4-c983-4616-9d88-64e3d3c3aefc",  # CHINESE & CONTINENTAL
    "https://angansweets.com/products?categoryId=5f5eb6f6-3f01-48f3-a99d-faeb9daef13b",  # CHATS
    "https://angansweets.com/products?categoryId=9283390d-e9e7-443a-aa7c-af4bcabcc31a",  # SNACKS
    "https://angansweets.com/products?categoryId=7452de18-fea1-4796-bdbf-46a059850a22",  # NAMKEENS
    "https://angansweets.com/products?categoryId=73807e47-70ad-4343-aac2-63144a211901",  # BAKERY
    "https://angansweets.com/products?categoryId=71f0e908-6b2b-43c2-801e-7fdd413dadaf",  # CAKES
]

def slugify(name):
    """Convert food name like 'Kaju Katli' → 'kaju_katli'"""
    name = name.lower().strip()
    name = re.sub(r'[^\w\s-]', '', name)   # remove special chars
    name = re.sub(r'[\s]+', '_', name)      # spaces to underscores
    return name

def get_product_links(category_url):
    """Scrape all product page links from a category page"""
    soup = BeautifulSoup(requests.get(category_url).text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/products/" in href and "categoryId" not in href:
            full_url = BASE_URL + href if href.startswith("/") else href
            links.append(full_url)
    return list(set(links))  # remove duplicates

def get_image_and_name(product_url):
    """From a product page, get the food name and real image URL"""
    soup = BeautifulSoup(requests.get(product_url).text, "html.parser")
    
    # Get food name from h1
    h1 = soup.find("h1")
    food_name = h1.get_text(strip=True) if h1 else None
    
    # Get image — find Next.js image with supabase URL
    img_url = None
    for img in soup.find_all("img"):
        src = img.get("src", "")
        if "supabase" in src or ("_next/image" in src and "logo" not in src):
            if "_next/image" in src:
                parsed = urlparse(src)
                params = parse_qs(parsed.query)
                if "url" in params:
                    real = unquote(params["url"][0])
                    if "logo" not in real:
                        img_url = real
                        break
            else:
                img_url = src
                break
    
    return food_name, img_url

Scraping Loop

In [2]:
os.makedirs("downloaded_images_v2", exist_ok=True)

all_products = []
seen_urls = set()

for cat_url in CATEGORY_URLS:
    print(f"\n📂 Scraping category: {cat_url.split('categoryId=')[1][:8]}...")
    product_links = get_product_links(cat_url)
    print(f"   Found {len(product_links)} products")
    
    for product_url in product_links:
        try:
            food_name, img_url = get_image_and_name(product_url)
            
            if not food_name or not img_url:
                print(f"   ⚠️ Skipping (missing name or image): {product_url}")
                continue
            
            if img_url in seen_urls:
                print(f"   ⏭️ Duplicate image, skipping: {food_name}")
                continue
            seen_urls.add(img_url)
            
            # Build clean filename from food name
            ext = img_url.split(".")[-1].split("?")[0].strip() or "jpg"
            filename = f"{slugify(food_name)}.{ext}"
            filepath = f"downloaded_images_v2/{filename}"
            
            # Download
            response = requests.get(img_url)
            with open(filepath, "wb") as f:
                f.write(response.content)
            
            all_products.append({
                "food_name": food_name,
                "filename": filename,
                "original_url": img_url,
                "product_url": product_url
            })
            print(f"   ✅ {food_name} → {filename}")
            
            time.sleep(0.2)  # be polite, don't hammer the server
        
        except Exception as e:
            print(f"   ❌ Error on {product_url}: {e}")

print(f"\n🎉 Done! {len(all_products)} images downloaded.")


📂 Scraping category: 56aa3726...
   Found 44 products
   ✅ Kesar Ghewar Big → kesar_ghewar_big.png
   ✅ Gondh Laddu → gondh_laddu.png
   ✅ Milk Cake → milk_cake.png
   ✅ Sugarfree Kaju Barfi → sugarfree_kaju_barfi.png
   ✅ Soan Papdi 400gm → soan_papdi_400gm.png
   ✅ Rasbhari → rasbhari.png
   ✅ Malai Ghewar Small → malai_ghewar_small.png
   ✅ Gund Pak 400gm → gund_pak_400gm.png
   ✅ Khoya Barfi → khoya_barfi.png
   ✅ Chocolate Barfi → chocolate_barfi.png
   ✅ Dhoda Barfi → dhoda_barfi.png
   ✅ Kaju Katli → kaju_katli.png
   ✅ Mota Boondi Laddu → mota_boondi_laddu.png
   ✅ Soan Papdi 200gm → soan_papdi_200gm.png
   ✅ Soan Haluwa 4pc → soan_haluwa_4pc.png
   ✅ Lal Mohan → lal_mohan.png
   ✅ Kesar Ghewar Small → kesar_ghewar_small.png
   ✅ Kalakand → kalakand.png
   ✅ Mathura Peda → mathura_peda.png
   ✅ Chandrakala → chandrakala.png
   ✅ Malai Ghewar Big → malai_ghewar_big.png
   ✅ Kanpuri Laddu → kanpuri_laddu.png
   ✅ Karachi Halwa → karachi_halwa.png
   ✅ Besan laddu → besan_laddu.p

Conversion

In [3]:
from PIL import Image
import os

os.makedirs("converted_images_v2", exist_ok=True)

for filename in os.listdir("downloaded_images_v2"):
    input_path = f"downloaded_images_v2/{filename}"
    output_filename = os.path.splitext(filename)[0] + ".webp"
    output_path = f"converted_images_v2/{output_filename}"

    try:
        img = Image.open(input_path)
        img.save(output_path, "WEBP", quality=85)
        print(f"✅ {filename} → {output_filename}")
    except Exception as e:
        print(f"❌ Skipping {filename} — {e}")

print(f"\n🎉 Done converting!")

✅ 2_jar_jaipuria_box_maroon.jpg → 2_jar_jaipuria_box_maroon.webp
✅ 2_jar_jaipuria_box_navy.jpg → 2_jar_jaipuria_box_navy.webp
✅ 3_jar_jaipuria_box_maroon.jpg → 3_jar_jaipuria_box_maroon.webp
✅ 3_jar_jaipuria_box_navy.jpg → 3_jar_jaipuria_box_navy.webp
✅ 3_jar_jaipuria_box_sky_blue.jpg → 3_jar_jaipuria_box_sky_blue.webp
✅ 3_line_assorted_sweet_box.png → 3_line_assorted_sweet_box.webp
✅ 3_line_sugarless_sweet_box.png → 3_line_sugarless_sweet_box.webp
✅ 4_chamber_golden_box.jpg → 4_chamber_golden_box.webp
✅ 4_chamber_red_velvet_boxdiamond.jpg → 4_chamber_red_velvet_boxdiamond.webp
✅ 4_chamber_red_velvet_boxsquare.jpg → 4_chamber_red_velvet_boxsquare.webp
✅ 4_chamber_royal_chestbronze.jpg → 4_chamber_royal_chestbronze.webp
✅ 4_chamber_royal_chestsilver.jpg → 4_chamber_royal_chestsilver.webp
✅ 4_khana_royal_thali.jpg → 4_khana_royal_thali.webp
✅ 4_line_assorted_sweet_box.png → 4_line_assorted_sweet_box.webp
✅ 4_line_mix_square_box_medium.png → 4_line_mix_square_box_medium.webp
✅ 5_khana_roy

Updated Data Map

In [10]:
import json

# Save from the in-memory variable (still alive in your kernel)
with open("data_map_v2.json", "w", encoding="utf-8") as f:
    json.dump(all_products, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_products)} products to data_map_v2.json")

Saved 248 products to data_map_v2.json


In [11]:
for item in all_products:
    item["converted_filename"] = os.path.splitext(item["filename"])[0] + ".webp"

with open("data_map_v2.json", "w", encoding="utf-8") as f:
    json.dump(all_products, f, indent=2, ensure_ascii=False)

print("data_map_v2.json updated with converted filenames!")

data_map_v2.json updated with converted filenames!
